In [92]:
import os
import math
from collections import defaultdict
from typing import Dict, Tuple, List, Optional
import csv
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.backends.backend_pdf import PdfPages


# define paths for datasets
data_root = r"/path/to/slice/data"
norm_file = r"/path/to/normalization.csv"
group_file = r"/path/to/groups.csv"
stroke_sides = r"/path/to/stroke_sides.csv"
output_folder = r"/path/to/output"

## Parameters from microscope

In [93]:
cell_size = 300 # grid cell size from microscope software
rotate_90_ccw = True # needed to rotate coordinates 90 degrees (to match output of software mto mouse anatomy picture)
export_group_svgs = True # plot each group as seperate .svg

## Data loading

In [94]:
def load_normalization(filepath: str) -> Dict[str, float]:
    """Loads animal-specific normalization factors from a CSV file."""
    data = {}
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader)  # Skip the header row
            for row in reader:
                if not row: continue
                animal_id, factor = row
                data[animal_id.strip()] = float(factor)
        print(f"Loaded {len(data)} normalization factors from {filepath}")
    except FileNotFoundError:
        print(f"[Error] Normalization file not found: {filepath}")
    except Exception as e:
        print(f"[Error] Failed to parse {filepath}: {e}")
    return data

def load_groups(filepath: str) -> Dict[str, List[int]]:
    """Loads group definitions from a CSV file."""
    data = {}
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader)  # Skip the header row
            for row in reader:
                if not row: continue
                group_name = row[0].strip()
                # Filter out any empty strings that might result from trailing commas
                members = [int(m.strip()) for m in row[1:] if m.strip()]
                data[group_name] = members
        print(f"Loaded {len(data)} groups from {filepath}")
    except FileNotFoundError:
        print(f"[Error] Groups file not found: {filepath}")
    except Exception as e:
        print(f"[Error] Failed to parse {filepath}: {e}")
    return data

def load_stroke_side(filepath: str) -> Dict[str, Optional[str]]:
    """Loads mirror axis settings from a CSV file."""
    data = {}
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader)  # Skip the header row
            for row in reader:
                if not row: continue
                animal_id = row[0].strip()
                stroke_side = row[1].strip
                data[animal_id] = stroke_side
        print(f"Loaded {len(data)} stroke side settings from {filepath}")
    except FileNotFoundError:
        print(f"[Error] Stroke side settings file not found: {filepath}")
    except Exception as e:
        print(f"[Error] Failed to parse {filepath}: {e}")
    return data

animal_to_normalization = load_normalization(norm_file)
groups = load_groups(group_file)
mirror_axis_map = load_stroke_side(stroke_sides)

print(f"\nGroups loaded: {list(groups.keys())}")

Loaded 22 normalization factors from F:\Chris\Projects\Nina\data\normalization.csv
Loaded 3 groups from F:\Chris\Projects\Nina\data\groups.csv
Loaded 22 stroke side settings from F:\Chris\Projects\Nina\data\stroke_sides.csv

Groups loaded: ['Sham', 'Stroke', 'Stroke_plus_Rehab']


## Helper functions for data processing and plotting

In [95]:
# =========================
# Label Helpers
# =========================
def _to_excel_letters(zero_based_idx: int) -> str:
    """Converts a zero-based index to an Excel-style column label (0->A, 26->AA)."""
    n = zero_based_idx + 1
    s = []
    while n > 0:
        n, r = divmod(n - 1, 26)
        s.append(chr(65 + r))
    return ''.join(reversed(s))


def get_column_idx_label(ci: int) -> str:
    """Gets the Excel-style label for a given column index."""
    if ci >= 0:
        return _to_excel_letters(ci)
    else:
        return '-' + _to_excel_letters(abs(ci) - 1)

# =========================
# Data Loading and Transformation
# =========================
def read_coordinates(file_path: str) -> List[Tuple[float, float]]:
    """Reads (x, y) coordinates from a marker file."""
    coords = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith(';'):
                    continue
                parts = line.split()
                # Expects x in the 2nd column and y in the 3rd.
                if len(parts) >= 3:
                    coords.append((float(parts[1]), float(parts[2])))
    except (IOError, ValueError, IndexError) as e:
        print(f"[Warning] Could not read or parse file {file_path}: {e}")
    return coords


def mirror_coords_about_grid_center_y(coords: List[Tuple[float, float]]) -> List[Tuple[float, float]]:
    """
    Mirrors coordinates vertically --> needs to be done for mice that have a mirrored stroke side
    The mirror axis is calculated based on the center of the resulting grid,
    ensuring the grid itself is symmetric rather than the raw coordinate space.
    Uses the global cell_size constant.
    """
    if not coords:
        return []

    rjs = [-math.ceil(y / cell_size) for _, y in coords]
    min_rj, max_rj = min(rjs), max(rjs)
    y_axis_twice = -cell_size * (min_rj + max_rj + 1)
    return [(x, y_axis_twice - y) for (x, y) in coords]


def rotate_coords_90_ccw_about_center(coords: List[Tuple[float, float]]) -> List[Tuple[float, float]]:
    """
    Rotates coordinates 90 degrees clockwise about their geometric center.
    """
    if not coords:
        return []
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    cx, cy = 0.5 * (min(xs) + max(xs)), 0.5 * (min(ys) + max(ys))

    rotated_coords = []
    for x, y in coords:
        x0, y0 = x - cx, y - cy
        xr, yr = -y0, x0
        rotated_coords.append((xr + cx, yr + cy))
    return rotated_coords


def count_cells_numeric(coords: List[Tuple[float, float]]) -> Dict[Tuple[int, int], int]:
    """
    Bins (x, y) coordinates into a grid and counts points per cell.
    Grid indices (ci, rj) are calculated as:
      ci = floor(x / S)  (column index)
      rj = -ceil(y / S)  (row index, inverted y-axis)
    Uses the global CELL_SIZE constant.
    """
    grid = defaultdict(int)
    for x, y in coords:
        # Note: Relies on the global CELL_SIZE
        ci = math.floor(x / cell_size)
        rj = -math.ceil(y / cell_size)
        grid[(ci, rj)] += 1
    return grid

# =========================
# Aggregation and Analysis
# =========================
def compute_global_bounds_and_values(dicts_of_counts: List[Dict[Tuple[int, int], float]]
                                      ) -> Tuple[Tuple[int, int, int, int], List[float]]:
    """
    Calculates the min/max grid indices and all count values across multiple dictionaries.
    """
    all_vals, all_cis, all_rjs = [], [], []
    for d in dicts_of_counts:
        if d:
            all_vals.extend(d.values())
            all_cis.extend(key[0] for key in d.keys())
            all_rjs.extend(key[1] for key in d.keys())

    if not all_vals:
        return (0, 0, 0, 0), [0.0]

    bounds = (min(all_cis), max(all_cis), min(all_rjs), max(all_rjs))
    return bounds, all_vals


# =========================
# Plotting
# =========================
def plot_heatmap(counts_numeric: Dict[Tuple[int, int], float],
                  title: str,
                  save_path: Optional[str] = None,
                  bounds: Optional[Tuple[int, int, int, int]] = None,
                  norm: Optional[Normalize] = None) -> Optional[plt.Figure]:
    """Generates and saves a single heatmap figure."""
    if not counts_numeric:
        print(f"[Info] No data to plot for: {title}")
        return None

    if bounds is None:
        cis, rjs = zip(*counts_numeric.keys())
        min_ci, max_ci = min(cis), max(cis)
        min_rj, max_rj = min(rjs), max(rjs)
    else:
        min_ci, max_ci, min_rj, max_rj = bounds

    width = max_ci - min_ci + 1
    height = max_rj - min_rj + 1

    grid = np.zeros((height, width), dtype=float)
    for (ci, rj), cnt in counts_numeric.items():
        if (min_ci <= ci <= max_ci) and (min_rj <= rj <= max_rj):
            # grid is indexed by [row, column]
            grid[rj - min_rj, ci - min_ci] = cnt

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(grid, origin='lower', norm=norm, cmap='viridis')

    # --- Ticks and Labels ---
    ax.set_xticks(np.arange(width))
    ax.set_xticklabels([get_column_idx_label(ci) for ci in range(min_ci, max_ci + 1)], rotation=90)
    ax.set_yticks(np.arange(height))
    ax.set_yticklabels([str(rj) for rj in range(min_rj, max_rj + 1)])

    ax.set_xlabel('Column Index')
    ax.set_ylabel('Row Index')
    ax.set_title(title, fontsize=16)

    cbar = fig.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label('Normalized Fiber Count Ratio')
    fig.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, bbox_inches='tight')

    return fig

print("All helper functions defined.")

All helper functions defined.


## Main Execution

In [96]:
print("--- Starting Step 4: Processing All Animals ---")
os.makedirs(output_folder, exist_ok=True)

mouse_to_agg_counts = {}

for mouse_id in os.listdir(data_root):
    animal_dir = os.path.join(data_root, mouse_id)
    if not os.path.isdir(animal_dir):
        continue

    norm_factor = animal_to_normalization.get(mouse_id)
    if not norm_factor:
        print(f"[Skip] {mouse_id}: Missing or zero normalization factor.")
        continue

    print(f"Processing animal: {mouse_id}")
    counts_per_cut = {}
    for filename in os.listdir(animal_dir):
        if not filename.lower().endswith(".txt"):
            continue

        file_path = os.path.join(animal_dir, filename)
        coords = read_coordinates(file_path)

        # Apply transformations
        if mirror_axis_map.get(mouse_id) == 'l':
            coords = mirror_coords_about_grid_center_y(coords)
        if rotate_90_ccw:
            coords = rotate_coords_90_ccw_about_center(coords)

        # Bin and normalize
        num_counts = count_cells_numeric(coords)
        counts_per_cut[filename] = {k: v / norm_factor for k, v in num_counts.items()}

    if not counts_per_cut:
        print(f"[Warning] No valid .txt cuts processed for mouse {mouse_id}.")
        continue

    # Aggregate all slices for this mouse by averaging
    all_cells = set.union(*[set(d.keys()) for d in counts_per_cut.values()])
    agg_counts = {
        cell: np.mean([d.get(cell, 0.0) for d in counts_per_cut.values()])
        for cell in all_cells
    }
    mouse_to_agg_counts[mouse_id] = agg_counts

print(f"--- Finished Processing. Found data for {len(mouse_to_agg_counts)} animals. ---")


--- Starting Step 4: Processing All Animals ---
Processing animal: 39
Processing animal: 42
Processing animal: 45
Processing animal: 52
Processing animal: 53
Processing animal: 54
Processing animal: 55
Processing animal: 56
Processing animal: 58
Processing animal: 59
Processing animal: 60
Processing animal: 61
Processing animal: 63
Processing animal: 64
Processing animal: 65
Processing animal: 68
Processing animal: 72
Processing animal: 76
Processing animal: 79
Processing animal: 80
Processing animal: 82
Processing animal: 84
--- Finished Processing. Found data for 22 animals. ---


## Aggregate by group

In [97]:
print("--- Starting Step 5: Aggregating by Group ---")

group_maps = {}
for group_name, mouse_ids in groups.items():
    print(f"Aggregating group: {group_name}")

    # Get all valid aggregated maps for mice in this group
    valid_mouse_maps = [mouse_to_agg_counts[str(mid)] for mid in mouse_ids if str(mid) in mouse_to_agg_counts]

    if not valid_mouse_maps:
        print(f"[Warning] Group '{group_name}' has no valid data. Skipping.")
        continue

    # Create a union of all cell coordinates present in the group
    group_cells = set.union(*[set(d.keys()) for d in valid_mouse_maps])

    # Calculate the mean for each cell, filling with 0 for mice that lack data in that cell
    group_agg = {
        cell: np.mean([m.get(cell, 0.0) for m in valid_mouse_maps])
        for cell in group_cells
    }
    group_maps[group_name] = group_agg

print(f"--- Finished Aggregation. Created {len(group_maps)} group maps. ---")

--- Starting Step 5: Aggregating by Group ---
Aggregating group: Sham
Aggregating group: Stroke
Aggregating group: Stroke_plus_Rehab
--- Finished Aggregation. Created 3 group maps. ---


## Generate and Save plots

In [98]:
if not group_maps:
    print("[Error] No group maps were generated. Cannot create plots. Exiting.")
else:
    print("--- Starting Step 6: Generating and Saving Plots ---")

    # --- Create a shared color scale and bounds for all group plots ---
    bounds, all_vals = compute_global_bounds_and_values(list(group_maps.values()))


    # --- Generate PDF and optional SVGs ---
    pdf_path = os.path.join(output_folder, "group_heatmaps.pdf")
    with PdfPages(pdf_path) as pdf:
        for group_name, counts in group_maps.items():
            print(f"Plotting group: {group_name}...")

            global_vmax = max(all_vals) if all_vals else 1.0
            shared_norm = Normalize(vmin=0.0, vmax=global_vmax)
            svg_path = None
            if export_group_svgs:
                svg_path = os.path.join(output_folder, f"{group_name}_agg_heatmap.svg")

            fig = plot_heatmap(
                counts,
                title=f"{group_name} Aggregate Heatmap",
                save_path=svg_path,
                bounds=bounds,  # Use the final, potentially clamped bounds
                norm=shared_norm,
            )
            if fig:
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)  # Close the figure to free up memory

            if svg_path and fig:
                print(f"  -> Saved {svg_path}")

    print(f"\n--- Pipeline Finished ---")
    print(f"Successfully created group PDF: {pdf_path}")

--- Starting Step 6: Generating and Saving Plots ---
Plotting group: Sham...
  -> Saved F:\Chris\Projects\Nina\data\output\Sham_agg_heatmap.svg
Plotting group: Stroke...
  -> Saved F:\Chris\Projects\Nina\data\output\Stroke_agg_heatmap.svg
Plotting group: Stroke_plus_Rehab...
  -> Saved F:\Chris\Projects\Nina\data\output\Stroke_plus_Rehab_agg_heatmap.svg

--- Pipeline Finished ---
Successfully created group PDF: F:\Chris\Projects\Nina\data\output\group_heatmaps.pdf
